In [56]:
import pandas as pd
import numpy as np
import time
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier

from scipy.sparse import hstack, csr_matrix
from sentence_transformers import SentenceTransformer

# Reading The File

In [4]:

df = pd.read_csv("tickets_modified_with_priority.csv")
print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset Shape: (2200, 5)
Columns: ['ticket_id', 'ticket_text', 'tenure', 'department', 'priority']


# Data Cleaning

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["ticket_text"] = df["ticket_text"].apply(clean_text)

# Encoding And Value Counts

In [6]:
print("\nDepartment distribution:")
print(df["department"].value_counts())

print("\nPriority distribution:")
print(df["priority"].value_counts())

# Encoding 
dept_encoder = LabelEncoder()
priority_encoder = LabelEncoder()

y_dept = dept_encoder.fit_transform(df["department"])
y_priority = priority_encoder.fit_transform(df["priority"])

# Scaling 
scaler = StandardScaler()
X_num = scaler.fit_transform(df[["tenure"]]) 


Department distribution:
department
Tech        472
Feedback    451
Account     440
Sales       437
Billing     400
Name: count, dtype: int64

Priority distribution:
priority
Medium    1666
High       298
Low        236
Name: count, dtype: int64


# TF-IDF Process

In [7]:
print("\nCreating TF-IDF features...")

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_text = tfidf.fit_transform(df["ticket_text"])

X_sparse = hstack([X_text, csr_matrix(X_num)]) 


Creating TF-IDF features...


# TF-IDF For Deprtment

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sparse, y_dept, test_size=0.2, stratify=y_dept, random_state=42
)

model = LinearSVC(class_weight="balanced")
model.fit(X_train, y_train)
preds = model.predict(X_test)

model_tree = DecisionTreeClassifier(max_depth=None,
                                    random_state=42)

model_rf = RandomForestClassifier(n_estimators=100, 
                                  max_depth=12, 
                                  random_state=42) 
print("\n --------TF-IDF (Department) --------")
print("Accuracy (LinearSVC):", round(accuracy_score(y_test, preds), 4)) 

model_tree.fit(X_train, y_train)
preds_tree = model_tree.predict(X_test)
print("Accuracy (DecisionTree):", round(accuracy_score(y_test, preds_tree), 4))

model_rf.fit(X_train, y_train)
preds_rf = model_rf.predict(X_test)
print("Accuracy (RandomForest):", round(accuracy_score(y_test, preds_rf), 4))


 --------TF-IDF (Department) --------
Accuracy (LinearSVC): 0.9523
Accuracy (DecisionTree): 0.7795
Accuracy (RandomForest): 0.9432


# TF-IDF For Priority

In [66]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sparse, y_priority, test_size=0.2, stratify=y_priority, random_state=42
)

model = LinearSVC(class_weight="balanced")
model.fit(X_train, y_train)
preds = model.predict(X_test)

model_tree = DecisionTreeClassifier(
    max_depth=None,
    random_state=32
) 
model_tree.fit(X_train, y_train)
preds_tree = model_tree.predict(X_test)

model_rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=None)
model_rf.fit(X_train, y_train)
preds_rf = model_rf.predict(X_test)

print("\n------TF-IDF (Priority) ------")
print("Accuracy(LinearSVC):", round(accuracy_score(y_test, preds), 4))
print("Accuracy (DecisionTree):", round(accuracy_score(y_test, preds_tree), 4))
print("Accuracy (RandomForest):", round(accuracy_score(y_test, preds_rf), 4)) 



------TF-IDF (Priority) ------
Accuracy(LinearSVC): 0.9455
Accuracy (DecisionTree): 1.0
Accuracy (RandomForest): 0.9841


# Some Random Predictions

In [31]:
new_text = "I just love your service"
new_tenure = 12

# Clean
cleaned_text = re.sub(r"[^a-zA-Z0-9 ]", " ", new_text.lower())
cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

# Transform
text_vector = tfidf.transform([cleaned_text])
tenure_scaled = scaler.transform([[new_tenure]])

# Combine
final_input = hstack([text_vector, csr_matrix(tenure_scaled)])

# Predict
dept_pred = dept_encoder.inverse_transform(model.predict(final_input))
priority_pred = priority_encoder.inverse_transform(model.predict(final_input))

print("Predicted Department:", dept_pred[0])
print("Predicted Priority:", priority_pred[0])

Predicted Department: Feedback
Predicted Priority: Medium


c:\Users\pauld\anaconda3\envs\debo\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


# Using ANN (Depatment)

### Not using Keras

In [27]:
X_dense = X_sparse.toarray()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_dense, y_dept, test_size=0.2, stratify=y_dept, random_state=42
)

model = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=20, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("\n------ ANN (Department) ------")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))



====== ANN (Department) ======
Accuracy: 0.9523


c:\Users\pauld\anaconda3\envs\debo\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


### Using Keras

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_dense, y_dept, test_size=0.2, stratify=y_dept, random_state=42
)

# Build model
dept_model_ANN = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(len(np.unique(y_dept)), activation='softmax')
])

dept_model_ANN.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train
dept_model_ANN.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)

# Evaluate
loss, accuracy = dept_model_ANN.evaluate(X_test, y_test, verbose=0)

print("\n------ ANN (Keras - Department) ------")
print("Accuracy:", round(accuracy, 4))

Epoch 1/5
55/55 [==============================] - 1s 4ms/step - loss: 1.5735 - accuracy: 0.3551
Epoch 2/5
55/55 [==============================] - 0s 3ms/step - loss: 1.1607 - accuracy: 0.7676
Epoch 3/5
55/55 [==============================] - 0s 3ms/step - loss: 0.4468 - accuracy: 0.9409
Epoch 4/5
55/55 [==============================] - 0s 2ms/step - loss: 0.2639 - accuracy: 0.9494
Epoch 5/5
55/55 [==============================] - 0s 2ms/step - loss: 0.2133 - accuracy: 0.9500

====== ANN (Keras - Department) ======
Accuracy: 0.9545


# Using ANN (Priority)

### Not Using Keras

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X_dense, y_priority, test_size=0.2, stratify=y_priority, random_state=42
)

model = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=20, random_state=42)


model.fit(X_train, y_train)
preds = model.predict(X_test)


print("\n====== ANN (Priority) ======")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))


====== ANN (Priority) ======
Accuracy: 0.9318


c:\Users\pauld\anaconda3\envs\debo\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


### Using Keras

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_dense, y_priority, test_size=0.2, stratify=y_priority, random_state=42
)

priority_model_ANN = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(len(np.unique(y_priority)), activation='softmax')
])

priority_model_ANN.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


priority_model_ANN.fit(X_train, y_train, epochs=5, batch_size=32, verbose=1)
loss, accuracy = priority_model_ANN.evaluate(X_test, y_test, verbose=0)

print("\n------ ANN (Keras - Priority) ------")
print("Accuracy:", round(accuracy, 4))


Epoch 1/5
55/55 [==============================] - 0s 4ms/step - loss: 0.8487 - accuracy: 0.7170
Epoch 2/5
55/55 [==============================] - 0s 4ms/step - loss: 0.5928 - accuracy: 0.7574
Epoch 3/5
55/55 [==============================] - 0s 2ms/step - loss: 0.4478 - accuracy: 0.7722
Epoch 4/5
55/55 [==============================] - 0s 2ms/step - loss: 0.2388 - accuracy: 0.8966
Epoch 5/5
55/55 [==============================] - 0s 2ms/step - loss: 0.0979 - accuracy: 0.9693

====== ANN (Keras - Priority) ======
Accuracy: 0.9341


# Some Random Prediction (ANN)

In [54]:
new_text = "what is the cost?"
new_tenure = 18

# Clean
cleaned_text = re.sub(r"[^a-zA-Z0-9 ]", " ", new_text.lower())
cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

# Transform
text_vector = tfidf.transform([cleaned_text])
tenure_scaled = scaler.transform([[new_tenure]])

# Combine
final_input = hstack([text_vector, csr_matrix(tenure_scaled)]).toarray()

# Department Prediction
dept_pred = dept_encoder.inverse_transform(
    np.argmax(dept_model_ANN.predict(final_input), axis=1)
)

# Priority Prediction
priority_pred = priority_encoder.inverse_transform(
    np.argmax(priority_model_ANN.predict(final_input), axis=1)
)

print("Predicted Department:", dept_pred[0])
print("Predicted Priority:", priority_pred[0])

1/1 [==============================] - 0s 55ms/step
Predicted Department: Billing
Predicted Priority: Low


c:\Users\pauld\anaconda3\envs\debo\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


# BERT Embedding

In [45]:
print("\nGenerating BERT embeddings...")

bert_model = SentenceTransformer("all-MiniLM-L6-v2")

start = time.time()
embeddings = bert_model.encode(
    df["ticket_text"].tolist(),
    batch_size=32,
    show_progress_bar=True
)
embed_time = time.time() - start

print("Embedding Time:", round(embed_time, 2), "sec")

X_bert = hstack([csr_matrix(embeddings), csr_matrix(X_num)])


Generating BERT embeddings...


Batches: 100%|██████████| 69/69 [00:02<00:00, 23.13it/s]


Embedding Time: 3.02 sec


### For Department



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bert, y_dept, test_size=0.2, stratify=y_dept, random_state=42
)

model = LogisticRegression(max_iter=300)

model.fit(X_train, y_train)
preds = model.predict(X_test)


print("\n------ BERT (Department) ------")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))


====== BERT (Department) ======
Accuracy: 0.9523


### For Priority

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bert, y_priority, test_size=0.2, stratify=y_priority, random_state=42
)

model = LogisticRegression(max_iter=300)

model.fit(X_train, y_train)
preds = model.predict(X_test)

print("\n------ BERT (Priority) ------")
print("Accuracy:", round(accuracy_score(y_test, preds), 4))


====== BERT (Priority) ======
Accuracy: 0.9568
